In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.ticker import PercentFormatter

In [ ]:
TASK = Path.cwd().parent.parent
INPUT, OUTPUT = TASK/'input', TASK/'output/exploratory'
OUTPUT.mkdir(parents=True,exist_ok=True)
CONFIG = json.loads((TASK.parent/'config.json').read_text())
Y = CONFIG['last_year']
MODELS = CONFIG['primary_plot_models']
LABELS = {'empirical_complete':'dblp', 'lognormal_grw':'GRW', 'ar1_s':'AR(1)',
          'hurdle_ar1_s_p':'H-AR(1)', 'se_hurdle_s':'SE-H'}
MARKERS = ['o','s','^','D','X']
LINES = ['-','--',':','-.',(0,(5,2,1,2,1,2))]
COLORS = ['.2',*sns.color_palette('colorblind',4)]

In [ ]:
curves = pd.read_csv(INPUT/'lognormal_residual_curves.csv')
summary = pd.read_csv(INPUT/'lognormal_residual_summary.csv')
display(summary)

In [ ]:
sns.set_theme(style='ticks',context='paper',rc={'pdf.fonttype':42})
fig,axes = plt.subplots(2,2,figsize=(8,5.5),sharex=True,sharey=True,layout='constrained')
empirical = curves.loc[curves.source.eq('empirical_complete')]
for i,(ax,source) in enumerate(zip(axes.flat,MODELS),start=1):
    model = curves.loc[curves.source.eq(source)]
    ax.axhline(0,color='.55',lw=.8)
    for frame,j in [(empirical,0),(model,i)]:
        ax.fill_between(frame.percentile,frame.lower,frame.upper,color=COLORS[j],alpha=.16,linewidth=0)
        sns.lineplot(data=frame,x='percentile',y='residual',color=COLORS[j],linestyle=LINES[j],
                     marker=MARKERS[j],markevery=12,markersize=3,estimator=None,errorbar=None,ax=ax,
                     label=LABELS[frame.source.iloc[0]])
    ax.legend(frameon=False)
    ax.set(xlabel='',ylabel='')
    ax.xaxis.set_major_formatter(PercentFormatter(1))
fig.supxlabel('Percentile of positive career totals')
fig.supylabel('Observed minus fitted log quantile (log-scale SD)')
fig.suptitle(f'Lognormal QQ residuals, career years {Y-3}–{Y}\nPointwise 95% matched-size bootstrap bands')
sns.despine(fig)
for ext in ['pdf','png']:
    fig.savefig(OUTPUT/f'lognormal_qq_residuals.{ext}',dpi=200)
plt.show()